# Ü-H — Lazy Evaluation: die zwei Fallen (LÖSUNG)

Spark ist lazy: Transformationen bauen nur den Plan, erst eine Aktion führt
aus. Falle 1: Mehrfach-Aktion ohne `cache` rechnet neu. Falle 2:
Transformation ohne Aktion tut nichts (kein Write, kein Fehler).

In [ ]:
%idle_timeout 60
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
# In Glue Studio wird die IAM-Rolle in der UI gewählt. Alternativ per Magic:
# %iam_role arn:aws:iam::<account>:role/AWSGlueServiceRole-GfuGlueTraining
%%configure
{ "--enable-glue-datacatalog": "true" }

In [ ]:
from awsglue.context import GlueContext
from pyspark.context import SparkContext
from pyspark.sql.functions import col, udf
from pyspark.sql.types import DoubleType

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

BUCKET = "gfu-glue-training-629452195361"

## Falle 1 — Neuberechnung ohne `cache`
Eine UDF mit Seiteneffekt (Print) macht sichtbar, wie oft Spark die Lineage
durchläuft. Zwei Aktionen auf demselben DataFrame → **zweimal** gerechnet.

In [ ]:
orders = glueContext.create_dynamic_frame.from_catalog(
    database="raw", table_name="orders", transformation_ctx="orders"
).toDF()

@udf(DoubleType())
def traced_double(x):
    print("  >> UDF berechnet eine Zeile")   # Seiteneffekt = Zähl-Marker
    return float(x) if x else 0.0

df = orders.withColumn("total", traced_double(col("order total")))

print("Aktion 1: count()")
df.count()
print("Aktion 2: show()  -> Marker erscheinen ERNEUT (Neuberechnung)")
df.show()

## Fix für Falle 1 — `cache()` + materialisieren
`cache()` ist selbst lazy; erst die **erste** Aktion füllt den Cache, danach
liest die zweite aus dem Speicher (keine Marker mehr).

In [ ]:
df.cache()
print("Materialisieren (1. Aktion, füllt den Cache):")
df.count()
print("2. Aktion aus dem Cache -> KEINE Marker:")
df.show()

## Falle 2 — Transformation ohne Aktion tut nichts
Ein `filter` allein löst nichts aus. Kein Ergebnis, aber auch **kein Fehler** —
die tückische Variante.

In [ ]:
print("nur Transformationen aufbauen ... (keine Ausgabe, keine Arbeit)")
filtered = df.filter(col("total") > 100)
print("Plan steht, aber nichts lief. Erst die Aktion zählt:")
print("Treffer:", filtered.count())

## Übertrag auf den Glue-Job
- `write` **ist** eine Aktion — löst die Pipeline aus. Fehlt jede Aktion, war der
  Job umsonst.
- `job.commit()` ist **keine** Spark-Aktion, aber der Abschluss, der Bookmarks
  fortschreibt. Ohne ihn verarbeitet der nächste Lauf dieselben Daten erneut
  (siehe Ü8.1).
- Mehrfach genutzte Zwischen-DataFrames (z. B. vor mehreren Sinks) → `cache()`,
  sonst rechnet Spark die teure Lineage je Sink neu.